Sequence length is the number of tokens in a single input example.
Example: “I love AI” → 3 tokens → sequence length = 3.

Input length usually means the same thing in practice: how many tokens are fed into the model at once. It can also refer to the maximum allowed size (like 512 tokens).

So:

* Sequence length → actual tokens in this input
* Input length → often same, but sometimes means model limit

In code (x.size(0) in PyTorch), it represents how many positions (rows) the model processes for that input.

In [1]:

import torch ## torch let's us create tensors and also provides helper functions
import torch.nn as nn ## torch.nn gives us nn.Module, nn.Embedding() and nn.Linear()
import torch.nn.functional as F # This gives us the softmax() and argmax()
from torch.optim import Adam # This is the optimizer we will use

import lightning as L # Lightning makes it easier to write, optimize and scale our code
from torch.utils.data import TensorDataset, DataLoader 

In [2]:
## first, a dictionary for the input vocabulary
input_vocab = {'<SOS>': 0, ## <SOS> = start of sequence.
               'lets': 1,
               'to': 2,
               'go': 3}

## Now a dictionary for the output vocabulary
output_vocab = {'<SOS>': 0,
                'ir': 1,
                'vamos': 2,
                'y': 3,
                '<EOS>': 4}

## Here are the english phrases, encoded using the
## input vocabulary
## NOTE: our transformer will prepend the <SOS> token to these inputs
inputs = torch.tensor([[1, 3],
                       [2, 3]])

## Here are the spanish translations encoded using
## the output vocabulary.
## NOTE: our transformer will prepend the <SOS> token to these outputs
labels = torch.tensor([[2],
                      [1]])

dataset = TensorDataset(inputs, labels) 
dataloader = DataLoader(dataset)


In [3]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model = 2, max_len = 3):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(start = 0, end = max_len, step = 1).float().unsqueeze(1)

        div_term = 1 / torch.tensor(10000.0) ** (torch.arange(start = 0, end = d_model, step = 2).float() / d_model)

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

## Attention Class

In [4]:
class Attention(nn.Module):
    def __init__(self, d_model=2):
        """
        d_model: int
            Dimensionality of token embeddings (features per token).
        """
        super().__init__()

        # Linear projections to generate Query (Q), Key (K), and Value (V)
        # Each maps from embedding space → embedding space
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection   = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)

        # Define which dimensions represent rows and columns
        # For input shape: (sequence_length, d_model)
        self.row_dim = 0   # tokens (positions)
        self.col_dim = 1   # embedding features

    def forward(self, query_input, key_input, value_input, mask=None):
        """
        query_input, key_input, value_input: Tensor
            Shape: (sequence_length, d_model)

        mask: Tensor or None
            Shape: (sequence_length, sequence_length)
            True values indicate positions to ignore (mask out).
        """

        # Step 1: Project inputs into Q, K, V spaces
        Q = self.query_projection(query_input)   # (seq_len, d_model)
        K = self.key_projection(key_input)       # (seq_len, d_model)
        V = self.value_projection(value_input)   # (seq_len, d_model)

        # Step 2: Compute raw attention scores
        # Formula: Q × K^T
        # Result shape: (seq_len, seq_len)
        attention_scores = torch.matmul(
            Q,
            K.transpose(self.row_dim, self.col_dim)
        )

        # Step 3: Scale attention scores
        # Formula: scores / sqrt(d_model)
        d_k = Q.size(self.col_dim)
        scaled_scores = attention_scores / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # Step 4: Apply mask (if provided)
        # Masked positions get large negative value → softmax ≈ 0
        if mask is not None:
            scaled_scores = scaled_scores.masked_fill(mask, -1e9)

        # Step 5: Convert scores to probabilities
        # Softmax over keys dimension (columns)
        attention_weights = F.softmax(scaled_scores, dim=self.col_dim)

        # Step 6: Weighted sum of values
        # Output shape: (seq_len, d_model)
        attention_output = torch.matmul(attention_weights, V)

        return attention_output

In [5]:
class Encoder(nn.Module):

    def __init__(self, num_tokens=4, d_model=2, max_len=3):
        """
        num_tokens: int
            Size of the vocabulary (number of unique token IDs).

        d_model: int
            Embedding dimension (features per token).

        max_len: int
            Maximum sequence length supported by positional encoding.
        """
        super().__init__()

        # Set seed for reproducibility (ensures same initial weights)
        L.seed_everything(seed=42)

        # Step 1: Token embedding layer
        # Maps token IDs → dense vectors
        # Input:  (seq_len,)
        # Output: (seq_len, d_model)
        self.token_embedding = nn.Embedding(
            num_embeddings=num_tokens,
            embedding_dim=d_model
        )

        # Step 2: Positional encoding
        # Adds deterministic position information
        self.position_encoding = PositionEncoding(
            d_model=d_model,
            max_len=max_len
        )

        # Step 3: Self-attention mechanism
        # Computes contextualized representations
        self.self_attention = Attention(d_model=d_model)

        # (Optional extension)
        # For multi-head attention:
        # - multiple Attention modules
        # - concatenate outputs
        # - project back to d_model

    def forward(self, token_ids):
        """
        token_ids: Tensor
            Shape: (sequence_length,)
            Each value is an integer index into the vocabulary.
        """

        # Step 1: Convert token IDs → embeddings
        # Shape: (seq_len, d_model)
        embeddings = self.token_embedding(token_ids)

        # Step 2: Add positional encoding
        # Shape: (seq_len, d_model)
        position_encoded_embeddings = self.position_encoding(embeddings)

        # Step 3: Apply self-attention
        # Q = K = V = same input (standard self-attention)
        # Output shape: (seq_len, d_model)
        attention_output = self.self_attention(
            position_encoded_embeddings,
            position_encoded_embeddings,
            position_encoded_embeddings
        )

        # Step 4: Residual connection
        # Combine input + attention output
        # Helps gradient flow and stabilizes training
        encoder_output = position_encoded_embeddings + attention_output

        return encoder_output

In [6]:
class Decoder(nn.Module):

    def __init__(self, num_tokens=4, d_model=2, max_len=3):
        """
        num_tokens: int
            Vocabulary size.

        d_model: int
            Embedding dimension.

        max_len: int
            Maximum sequence length.
        """
        super().__init__()

        # Set different seed from encoder to initialize different weights
        L.seed_everything(seed=43)

        # Step 1: Token embedding
        # Input:  (seq_len,)
        # Output: (seq_len, d_model)
        self.token_embedding = nn.Embedding(
            num_embeddings=num_tokens,
            embedding_dim=d_model
        )

        # Step 2: Positional encoding
        self.position_encoding = PositionEncoding(
            d_model=d_model,
            max_len=max_len
        )

        # Step 3: Masked self-attention (decoder-side)
        self.masked_self_attention = Attention(d_model=d_model)

        # Step 4: Encoder-Decoder attention
        # Q comes from decoder, K and V come from encoder
        self.cross_attention = Attention(d_model=d_model)

        # Step 5: Final projection to vocabulary space
        # Output: (seq_len, num_tokens)
        self.output_projection = nn.Linear(
            in_features=d_model,
            out_features=num_tokens
        )

        self.row_dim = 0
        self.col_dim = 1

    def forward(self, token_ids, encoder_outputs):
        """
        token_ids: Tensor
            Shape: (sequence_length,)
            Decoder input tokens.

        encoder_outputs: Tensor
            Shape: (sequence_length, d_model)
            Output from encoder.
        """

        # Step 1: Token embeddings
        embeddings = self.token_embedding(token_ids)

        # Step 2: Add positional encoding
        position_encoded = self.position_encoding(embeddings)

        # Step 3: Create causal mask (look-ahead mask)
        # Shape: (seq_len, seq_len)
        seq_len = token_ids.size(self.row_dim)

        # Lower triangular matrix (allowed positions = 1)
        causal_mask = torch.tril(torch.ones((seq_len, seq_len)))

        # Convert to boolean mask:
        # True  → masked (disallowed)
        # False → allowed
        causal_mask = causal_mask == 0

        # Step 4: Masked self-attention
        # Prevents attending to future tokens
        self_attention_output = self.masked_self_attention(
            position_encoded,
            position_encoded,
            position_encoded,
            mask=causal_mask
        )

        # Step 5: Residual connection (after masked self-attention)
        decoder_state = position_encoded + self_attention_output

        # Step 6: Encoder-Decoder (cross) attention
        # Query: decoder state
        # Key, Value: encoder outputs
        cross_attention_output = self.cross_attention(
            decoder_state,   # Q
            encoder_outputs, # K
            encoder_outputs  # V
        )

        # Step 7: Residual connection (after cross-attention)
        decoder_state = decoder_state + cross_attention_output

        # Step 8: Project to vocabulary logits
        # Shape: (seq_len, num_tokens)
        logits = self.output_projection(decoder_state)

        return logits

Masked self-attention lets the decoder read its own generated tokens while preventing access to future positions. This enforces causal generation—each token is predicted using only earlier tokens.

Cross-attention connects the decoder to the encoder output. It allows each decoder token to focus on relevant parts of the input sequence, bringing in contextual information.

Together:

* Masked self-attention → maintains correct generation order
* Cross-attention → injects source information

Without masking, the model would cheat. Without cross-attention, the decoder would ignore the input entirely and behave like a standalone language model.

In [7]:
class Transformer(L.LightningModule):

    def __init__(self, input_size, output_size, d_model=2, max_len=3):
        """
        input_size: int
            Size of input vocabulary.

        output_size: int
            Size of output vocabulary.

        d_model: int
            Embedding dimension.

        max_len: int
            Maximum sequence length.
        """
        super().__init__()

        # Encoder processes source (input) sequence
        self.encoder = Encoder(
            num_tokens=input_size,
            d_model=d_model,
            max_len=max_len
        )

        # Decoder generates target sequence
        self.decoder = Decoder(
            num_tokens=output_size,
            d_model=d_model,
            max_len=max_len
        )

        # Cross-entropy loss (applies softmax internally)
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, encoder_input_tokens, decoder_input_tokens):
        """
        encoder_input_tokens: (seq_len,)
        decoder_input_tokens: (seq_len,)

        Returns:
            logits: (seq_len, vocab_size)
        """

        # Step 1: Encode input sequence
        encoder_outputs = self.encoder(encoder_input_tokens)

        # Step 2: Decode using encoder outputs + previous tokens
        logits = self.decoder(decoder_input_tokens, encoder_outputs)

        return logits

    def configure_optimizers(self):
        # Optimizer for training
        return Adam(self.parameters(), lr=0.01)

    def training_step(self, batch, batch_idx):
        """
        batch: tuple(input_tensor, target_tensor)
        """

        input_tokens, target_tokens = batch  # unpack batch

        # Remove batch dimension (assuming batch_size = 1)
        input_tokens = input_tokens[0]
        target_tokens = target_tokens[0]

        # Step 1: Add <SOS> token (start of sequence)
        # Assume:
        # <SOS> = 0
        # <EOS> = 4
        encoder_input = torch.cat((torch.tensor([0]), input_tokens))

        # Decoder input uses teacher forcing:
        # shift target right and prepend <SOS>
        decoder_input = torch.cat((torch.tensor([0]), target_tokens))

        # Expected output:
        # shift target left and append <EOS>
        expected_output = torch.cat((target_tokens, torch.tensor([4])))

        # Step 2: Forward pass
        logits = self.forward(encoder_input, decoder_input)

        # Step 3: Compute loss
        # logits: (seq_len, vocab_size)
        # expected_output: (seq_len,)
        loss = self.criterion(logits, expected_output)

        return loss

In [8]:
# Step 0: Initialize Transformer model
max_length = 3
transformer = Transformer(
    input_size=len(input_vocab),
    output_size=len(output_vocab),
    d_model=2,
    max_len=max_length
)

# Step 1: Encode input sequence
# Example: <SOS> lets go → [0, 1, 3]
encoder_outputs = transformer.encoder(
    torch.tensor([0, 1, 3])
)

# Step 2: Initialize decoder input with <SOS>
# This acts as the first token for generation
generated_token_ids = torch.tensor([0])

# Step 3: Autoregressive decoding loop
for step in range(max_length):

    # Pass current generated sequence + encoder outputs to decoder
    # Output shape: (current_seq_len, vocab_size)
    logits = transformer.decoder(
        generated_token_ids,
        encoder_outputs
    )

    # Select next token:
    # - Take last time step (latest prediction)
    # - Apply argmax over vocabulary dimension
    next_token_id = torch.argmax(logits[-1, :]).unsqueeze(0)

    # Append predicted token to sequence
    generated_token_ids = torch.cat(
        (generated_token_ids, next_token_id)
    )

    # Stop if <EOS> token is generated
    if next_token_id.item() == 4:
        break

# Final generated sequence of token IDs
print("predicted_ids:", generated_token_ids)

Seed set to 42
Seed set to 43


predicted_ids: tensor([0, 2, 0, 1])


In [9]:
trainer = L.Trainer(max_epochs=30, accelerator="cpu")
trainer.fit(transformer, train_dataloaders=dataloader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/GoQuant/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/opt/anaconda3/envs/GoQuant/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Li

Epoch 29: 100%|█| 2/2 [00:00<00:00, 543.51it/s

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|█| 2/2 [00:00<00:00, 315.16it/s


In [11]:
# Step 0: Define maximum generation length
max_length = 3

# Step 1: Encode input sequence
# Input: <SOS> lets go → [0, 1, 3]
# Output: contextual representations (seq_len, d_model)
encoder_outputs = transformer.encoder(
    torch.tensor([0, 2, 3])
)

# Step 2: Initialize decoder input with <SOS>
# Shape: (1,)
generated_ids = torch.tensor([0])

# Step 3: Autoregressive decoding
for step in range(max_length):

    # Decoder forward pass
    # Input: all previously generated tokens + encoder outputs
    # Output: logits for each position → (current_len, vocab_size)
    logits = transformer.decoder(
        generated_ids,
        encoder_outputs
    )

    # Select next token:
    # - Use last position (latest time step)
    # - Choose token with highest logit (argmax)
    next_token = torch.argmax(logits[-1, :]).unsqueeze(0)

    # Append new token to sequence
    generated_ids = torch.cat((generated_ids, next_token))

    # Stop if <EOS> token is generated
    if next_token.item() == 4:
        break

# Final predicted sequence
print("predicted_ids:", generated_ids)

predicted_ids: tensor([0, 1, 4])


Here is your Transformer rewritten with technically precise comments, correct terminology, and clean variable naming, while preserving your logic.

⸻

class Transformer(L.LightningModule):
    def __init__(self, input_size, output_size, d_model=2, max_len=3):
        """
        input_size: int
            Size of input vocabulary.
        output_size: int
            Size of output vocabulary.
        d_model: int
            Embedding dimension.
        max_len: int
            Maximum sequence length.
        """
        super().__init__()
        # Encoder processes source (input) sequence
        self.encoder = Encoder(
            num_tokens=input_size,
            d_model=d_model,
            max_len=max_len
        )
        # Decoder generates target sequence
        self.decoder = Decoder(
            num_tokens=output_size,
            d_model=d_model,
            max_len=max_len
        )
        # Cross-entropy loss (applies softmax internally)
        self.criterion = nn.CrossEntropyLoss()
    def forward(self, encoder_input_tokens, decoder_input_tokens):
        """
        encoder_input_tokens: (seq_len,)
        decoder_input_tokens: (seq_len,)
        Returns:
            logits: (seq_len, vocab_size)
        """
        # Step 1: Encode input sequence
        encoder_outputs = self.encoder(encoder_input_tokens)
        # Step 2: Decode using encoder outputs + previous tokens
        logits = self.decoder(decoder_input_tokens, encoder_outputs)
        return logits
    def configure_optimizers(self):
        # Optimizer for training
        return Adam(self.parameters(), lr=0.1)
    def training_step(self, batch, batch_idx):
        """
        batch: tuple(input_tensor, target_tensor)
        """
        input_tokens, target_tokens = batch  # unpack batch
        # Remove batch dimension (assuming batch_size = 1)
        input_tokens = input_tokens[0]
        target_tokens = target_tokens[0]
        # Step 1: Add <SOS> token (start of sequence)
        # Assume:
        # <SOS> = 0
        # <EOS> = 4
        encoder_input = torch.cat((torch.tensor([0]), input_tokens))
        # Decoder input uses teacher forcing:
        # shift target right and prepend <SOS>
        decoder_input = torch.cat((torch.tensor([0]), target_tokens))
        # Expected output:
        # shift target left and append <EOS>
        expected_output = torch.cat((target_tokens, torch.tensor([4])))
        # Step 2: Forward pass
        logits = self.forward(encoder_input, decoder_input)
        # Step 3: Compute loss
        # logits: (seq_len, vocab_size)
        # expected_output: (seq_len,)
        loss = self.criterion(logits, expected_output)
        return loss

⸻

🔷 Full Training Flow (Architecture View)

Input Tokens
   ↓
+ <SOS>
   ↓
Encoder
   ↓
Encoder Outputs
   ↓
Decoder Input (Teacher Forcing: <SOS> + target)
   ↓
Decoder
   ↓
Logits (per token)
   ↓
Compare with (target + <EOS>)
   ↓
CrossEntropy Loss

⸻

🔷 Critical concepts

1. Teacher Forcing

* Decoder receives ground-truth tokens
* Not its own predictions
* Stabilizes and accelerates training

⸻

2. Token shifting

Purpose	Sequence
Encoder input	<SOS> + input
Decoder input	<SOS> + target
Expected output	target + <EOS>

This creates a one-step prediction offset.

⸻

3. Why logits (no softmax)?

* nn.CrossEntropyLoss() internally applies:

log_softmax + NLL loss

* So raw logits are required

⸻

🔷 Important issue in your original code

You used:

len(input_vocab)
len(output_vocab)

But passed:

input_size, output_size

This creates inconsistency. The corrected version uses:

num_tokens=input_size / output_size

⸻

🔷 One-line intuition

The model:

* encodes the input sequence
* decodes step-by-step using teacher forcing
* learns to predict the next token at each position

⸻

If you want, I can now simulate one full training example numerically so you see every tensor transformation step-by-step.

In [ ]:
Here is your inference loop rewritten with clear technical comments, correct terminology, and precise reasoning.

⸻

# Step 0: Initialize Transformer model
transformer = Transformer(
    input_size=len(input_vocab),
    output_size=len(output_vocab),
    d_model=2,
    max_len=max_length
)
# Step 1: Encode input sequence
# Example: <SOS> lets go → [0, 1, 3]
encoder_outputs = transformer.encoder(
    torch.tensor([0, 1, 3])
)
# Step 2: Initialize decoder input with <SOS>
# This acts as the first token for generation
generated_token_ids = torch.tensor([0])
# Step 3: Autoregressive decoding loop
for step in range(max_length):
    # Pass current generated sequence + encoder outputs to decoder
    # Output shape: (current_seq_len, vocab_size)
    logits = transformer.decoder(
        generated_token_ids,
        encoder_outputs
    )
    # Select next token:
    # - Take last time step (latest prediction)
    # - Apply argmax over vocabulary dimension
    next_token_id = torch.argmax(logits[-1, :]).unsqueeze(0)
    # Append predicted token to sequence
    generated_token_ids = torch.cat(
        (generated_token_ids, next_token_id)
    )
    # Stop if <EOS> token is generated
    if next_token_id.item() == 4:
        break
# Final generated sequence of token IDs
print("predicted_ids:", generated_token_ids)

⸻

🔷 What is happening (exact flow)

1. Encoder converts input tokens → encoder_outputs (context representation)
2. Decoder starts with <SOS>
3. At each step:
    * Takes all previously generated tokens
    * Uses masked self-attention (no future leakage)
    * Uses cross-attention (reads encoder outputs)
    * Predicts next token

⸻

🔷 Key technical insight

* logits[-1, :] → prediction for latest token only
* Earlier rows are recomputed because:
    * self-attention depends on entire sequence

⸻

🔷 One-line intuition

This is autoregressive generation:
the model repeatedly predicts the next token using all previously generated tokens plus encoder context.

⸻

If you want, I can map each tensor shape at every iteration—that reveals exactly how sequence length grows step by step.

Good—this deserves a structured, rigorous treatment. I’ll break it into multiple parts, each building toward a full system-level understanding of your Transformer implementation.

⸻

📘 Technical Report — Part 1

Foundations: Positional Encoding and Attention Mechanisms

⸻

1. Problem Context

The Transformer architecture introduced in Attention Is All You Need eliminates recurrence and convolution, relying entirely on attention mechanisms. This creates a fundamental issue:

The model has no inherent notion of token order.

Unlike RNNs (sequential) or CNNs (local receptive fields), attention treats inputs as a set, not a sequence.

⸻

2. Positional Encoding: Formal Definition

To inject order information, positional encodings are added to token embeddings:

\text{Input Representation} = \text{Embedding} + \text{Positional Encoding}

⸻

2.1 Mathematical Formulation

For position pos and dimension index i:

PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)

PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)

⸻

2.2 Key Properties

1. Deterministic (non-learned)
2. Continuous and smooth
3. Encodes relative position implicitly
4. Different frequency per dimension

⸻

3. Implementation Analysis: PositionEncoding

3.1 Tensor Construction

pe = torch.zeros(max_len, d_model)

Defines a matrix:

pe \in \mathbb{R}^{(\text{max\_len} \times d_{model})}

⸻

3.2 Position Vector

position = torch.arange(0, max_len).float().unsqueeze(1)

Shape:

(max\_len, 1)

⸻

3.3 Frequency Scaling

div_term = 1 / 10000^{(2i / d_model)}

This controls wavelength:

* lower dimensions → high frequency
* higher dimensions → low frequency

⸻

3.4 Encoding Assignment

pe[:, 0::2] = sin(position * div_term)
pe[:, 1::2] = cos(position * div_term)

This interleaves sine and cosine across dimensions.

⸻

3.5 Buffer Registration

self.register_buffer('pe', pe)

Ensures:

* not trainable
* stored with model
* moves across devices

⸻

3.6 Forward Pass

x + pe[:x.size(0)]

Injects positional information into embeddings.

⸻

4. Attention Mechanism

⸻

4.1 Core Idea

Each token computes a weighted combination of all tokens:

\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V

⸻

5. Implementation Analysis: Attention

⸻

5.1 Linear Projections

Q = W_q x
K = W_k x
V = W_v x

Each token is projected into three spaces:

* Query (what I want)
* Key (what I offer)
* Value (what I provide)

⸻

5.2 Similarity Computation

sims = torch.matmul(Q, K.T)

Produces:

(seq\_len, seq\_len)

Each element = similarity between token pairs.

⸻

5.3 Scaling

scaled = sims / sqrt(d_model)

Prevents:

* large dot products
* softmax saturation

⸻

5.4 Masking (optional)

scaled = scaled.masked_fill(mask, -1e9)

Used for:

* padding masking
* causal masking (decoder)

⸻

5.5 Softmax

weights = softmax(scaled)

Creates probability distribution over tokens.

⸻

5.6 Output

output = weights @ V

Each token becomes a weighted sum of values.

⸻

6. Critical Insight: Q, K, V Mapping

Your implementation defines:

Attention(q, k, v)

This is not inferred—it is explicitly controlled by argument order.

* First argument → Query
* Second → Key
* Third → Value

This design becomes crucial later in encoder-decoder attention.

⸻

7. Summary of Part 1

* Transformers require explicit positional encoding
* Sinusoidal encoding provides structured positional signals
* Attention computes contextual relationships using Q, K, V
* Scaling and masking stabilize training and enforce constraints
* Argument order defines semantic roles (Q, K, V)

⸻

Next, I’ll move to:

➡️ Part 2: Encoder Architecture and Data Flow

(including embedding pipeline, residual connections, and limitations of your simplified encoder)

⸻

Continue?

📘 Technical Report — Part 2

Encoder Architecture and Data Flow

⸻

1. Role of the Encoder

The encoder transforms an input token sequence into a contextual representation:

\text{Encoder Output} \in \mathbb{R}^{(seq\_len \times d_{model})}

Each output vector represents a token in context of all other tokens.

⸻

2. Input Representation Pipeline

2.1 Token IDs → Embeddings

embeddings = self.token_embedding(token_ids)

* Input: (seq_len,)
* Output: (seq_len, d_model)

Each token ID is mapped to a dense vector.

⸻

2.2 Add Positional Encoding

position_encoded = self.position_encoding(embeddings)

x = \text{Embedding} + \text{Positional Encoding}

Now each vector contains:

* semantic meaning
* positional information

⸻

3. Self-Attention in Encoder

3.1 Definition

attention_output = self.self_attention(x, x, x)

This is self-attention because:

Q = K = V = x

⸻

3.2 Interpretation

Each token:

* queries all tokens (including itself)
* computes attention weights
* aggregates information

Result:

\text{attention\_output} \in \mathbb{R}^{(seq\_len \times d_{model})}

⸻

4. Residual Connection

encoder_output = x + attention_output

⸻

4.1 Purpose

Residual connections:

1. Preserve original signal
2. Improve gradient flow
3. Prevent degradation in deep networks

⸻

4.2 Mathematical Form

y = x + \text{Attention}(x)

⸻

5. Full Encoder Flow

Token IDs
   ↓
Embedding Layer
   ↓
+ Positional Encoding
   ↓
Self-Attention (Q=K=V)
   ↓
+ Residual Connection
   ↓
Encoder Output

⸻

6. Shape Evolution

Stage	Shape
Token IDs	(seq_len,)
Embeddings	(seq_len, d_model)
+ Positional Encoding	(seq_len, d_model)
Self-Attention Output	(seq_len, d_model)
Final Output	(seq_len, d_model)

⸻

7. Conceptual Interpretation

After encoding:

Each token vector contains:

* its own meaning
* information from all other tokens
* weighted by relevance

Example:

Input:

<SOS> lets go

After encoding:

* “lets” may attend strongly to “go”
* “go” may attend to “lets”

Thus representations become context-aware.

⸻

8. Limitations of Your Encoder (Intentional Simplification)

Your implementation omits key Transformer components:

❌ 1. Layer Normalization

Normally:

LayerNorm(x + Attention(x))

Purpose:

* stabilize activations
* improve convergence

⸻

❌ 2. Feed Forward Network (FFN)

Standard Transformer adds:

FFN(x) = \max(0, xW_1 + b_1)W_2 + b_2

Adds:

* non-linearity
* feature transformation

⸻

❌ 3. Multi-Head Attention

Instead of one attention:

\text{MultiHead}(Q,K,V) = \text{Concat}(head_1,...,head_h)W^O

Allows:

* multiple representation subspaces
* richer relationships

⸻

❌ 4. Dropout

Used for regularization.

⸻

9. Why This Simplified Encoder Still Works

Despite missing components, your encoder:

* correctly implements attention mechanics
* demonstrates core idea of contextualization
* is sufficient for conceptual understanding

⸻

10. Critical Insight

The encoder produces:

encoder_outputs

These are later used as:

* Keys (K)
* Values (V)

in decoder cross-attention.

This design enables:

The decoder to “look up” relevant parts of the input sequence.

⸻

11. Summary of Part 2

* Encoder converts tokens → contextual embeddings
* Self-attention enables global interaction
* Residual connection preserves information
* Output is a context-rich representation
* Simplified version omits normalization, FFN, and multi-head attention

⸻

Next:

➡️ Part 3: Decoder Architecture, Masking, and Cross-Attention

(critical for understanding sequence generation and your Q/K/V confusion)

⸻

Continuing.

📘 Technical Report — Part 3

Decoder Architecture, Masking, and Cross-Attention

⸻

1. Role of the Decoder

The decoder generates the output sequence autoregressively, meaning:

y_t = f(y_{<t}, \text{encoder\_outputs})

Each token depends on:

* previously generated tokens
* encoder context

⸻

2. Decoder Input Pipeline

2.1 Token Embedding

embeddings = self.token_embedding(token_ids)

* Input: (seq_len,)
* Output: (seq_len, d_model)

⸻

2.2 Add Positional Encoding

position_encoded = self.position_encoding(embeddings)

Same mechanism as encoder.

⸻

3. Masked Self-Attention

⸻

3.1 Purpose

Prevent the decoder from accessing future tokens during training.

Without masking:

* model could “see the answer”
* breaks autoregressive property

⸻

3.2 Mask Construction

mask = torch.tril(torch.ones(seq_len, seq_len))
mask = mask == 0

Result (for seq_len=4):

Allowed (False):   Disallowed (True):
[[F, T, T, T],
 [F, F, T, T],
 [F, F, F, T],
 [F, F, F, F]]

⸻

3.3 Application in Attention

scaled_scores = scaled_scores.masked_fill(mask, -1e9)

Effect:

* masked positions → ~0 probability after softmax

⸻

3.4 Computation

self_attention_output = Attention(x, x, x, mask)

Q = K = V = \text{decoder input}

⸻

4. First Residual Connection

decoder_state = position_encoded + self_attention_output

Preserves original representation.

⸻

5. Cross-Attention (Encoder–Decoder Attention)

This is the most critical conceptual step.

⸻

5.1 Implementation

cross_attention_output = self.cross_attention(
    decoder_state,     # Q
    encoder_outputs,   # K
    encoder_outputs    # V
)

⸻

5.2 Why This Mapping?

Because your Attention is defined as:

Attention(q, k, v)

So:

Argument Position	Meaning	Source
1st	Q	Decoder
2nd	K	Encoder
3rd	V	Encoder

⸻

5.3 Conceptual Interpretation

* Decoder asks: “What do I need?” → Q
* Encoder provides: “What information exists?” → K, V

⸻

5.4 Mathematical Form

\text{Attention}(Q_{dec}, K_{enc}, V_{enc})

This allows each decoder token to:

* attend to relevant encoder tokens
* extract aligned information

⸻

5.5 Example Intuition

Input:

<SOS> lets go

Output target:

<SOS> vamos <EOS>

When predicting “vamos”:

* decoder queries encoder outputs
* attends strongly to “lets go”

⸻

6. Second Residual Connection

decoder_state = decoder_state + cross_attention_output

⸻

7. Output Projection

logits = self.output_projection(decoder_state)

Shape:
(seq\_len, vocab\_size)

Each row = prediction distribution over vocabulary.

⸻

8. No Softmax Here (Important)

You correctly avoided:

softmax(logits)

Because:

\text{CrossEntropyLoss} = \text{log\_softmax} + \text{NLL}

⸻

9. Full Decoder Flow

Decoder Input Tokens
        ↓
Embedding
        ↓
+ Positional Encoding
        ↓
Masked Self-Attention
        ↓
+ Residual
        ↓
Cross-Attention (Encoder)
        ↓
+ Residual
        ↓
Linear Projection
        ↓
Logits

⸻

10. Key Differences: Encoder vs Decoder

Component	Encoder	Decoder
Self-attention	Yes	Yes (masked)
Masking	No	Yes
Cross-attention	No	Yes
Input dependency	Input only	Input + previous outputs

⸻

11. Critical Insight (Your Core Confusion)

You asked:

How do we know Q is decoder and K,V are encoder?

Answer:

👉 It is defined explicitly by how you pass arguments.

self.cross_attention(decoder_state, encoder_outputs, encoder_outputs)

There is:

* no automatic inference
* no hidden mechanism

The mapping is purely positional in function arguments.

⸻

12. Summary of Part 3

* Masked self-attention enforces autoregressive behavior
* Cross-attention connects decoder to encoder
* Q comes from decoder because it is passed first
* K, V come from encoder because they are passed second and third
* Residual connections stabilize learning

⸻

Next:

➡️ Part 4: Full Transformer (Encoder + Decoder + Training Logic)

including teacher forcing, token shifting, and loss computation

⸻

Continuing.

📘 Technical Report — Part 4

Full Transformer Pipeline: Training Logic, Teacher Forcing, and Loss

⸻

1. System Overview

Your Transformer class integrates:

* Encoder → processes input sequence
* Decoder → generates output sequence
* Loss → supervises predictions

Formally:

\text{Output} = \text{Decoder}(\text{Target Input}, \text{Encoder(Input)})

⸻

2. Forward Pass

encoder_outputs = self.encoder(encoder_input_tokens)
logits = self.decoder(decoder_input_tokens, encoder_outputs)

⸻

2.1 Key Observation

* Encoder runs once per input
* Decoder runs conditioned on encoder output + partial target

⸻

3. Training Data Preparation

Your training step constructs three sequences:

⸻

3.1 Encoder Input

encoder_input = <SOS> + input_tokens

Example:

input:        lets go
encoder_input: <SOS> lets go

⸻

3.2 Decoder Input (Teacher Forcing)

decoder_input = <SOS> + target_tokens

Example:

target:         vamos
decoder_input:  <SOS> vamos

⸻

3.3 Expected Output

expected_output = target_tokens + <EOS>

Example:

expected_output: vamos <EOS>

⸻

4. Why This Shifting Works

This creates a one-step prediction alignment:

Time Step	Decoder Input	Expected Output
t=0	<SOS>	vamos
t=1	vamos	<EOS>

So the model learns:

P(y_t | y_{<t}, x)

⸻

5. Teacher Forcing

⸻

5.1 Definition

During training:

* Decoder receives ground truth tokens, not predictions

⸻

5.2 Why It’s Needed

Without teacher forcing:

* errors accumulate early
* training becomes unstable

⸻

5.3 In Your Code

decoder_input = torch.cat((<SOS>, target_tokens))

This ensures correct conditioning.

⸻

6. Loss Computation

⸻

6.1 Logits

Output of decoder:

\text{logits} \in \mathbb{R}^{(seq\_len \times vocab\_size)}

⸻

6.2 Target

\text{expected\_output} \in \mathbb{R}^{(seq\_len)}

⸻

6.3 Cross-Entropy Loss

loss = CrossEntropyLoss(logits, expected_output)

Internally:

\text{Loss} = -\log P(y_{true})

⸻

7. Important Shape Requirement

Tensor	Shape
logits	(N, C)
expected_output	(N,)

Where:

* N = seq\_len
* C = vocab\_size

⸻

8. Training Flow (End-to-End)

Input Tokens
   ↓
+ <SOS>
   ↓
Encoder
   ↓
Encoder Outputs
   ↓
Decoder Input (<SOS> + target)
   ↓
Decoder
   ↓
Logits
   ↓
Compare with (target + <EOS>)
   ↓
CrossEntropy Loss

⸻

9. Optimization

Adam(self.parameters(), lr=...)

⸻

9.1 Critical Issue You Encountered

You used:

lr = 0.1

This is extremely high for Adam.

Effects:

* exploding gradients
* NaN values
* kernel crash

⸻

9.2 Correct Range

lr = 1e-3  # standard

⸻

10. Lightning Training Loop

Using PyTorch Lightning:

trainer.fit(model, dataloader)

Lightning handles:

* forward pass
* backpropagation
* optimizer step

⸻

11. Kernel Crash Analysis

From your session:

Most probable causes:

1. High learning rate
2. NaN/Inf in attention
3. Shape mismatch in loss
4. DataLoader issues
5. GPU misconfiguration

⸻

12. Debug Strategy

Minimal checks:

print(logits.shape, expected_output.shape)
torch.isnan(logits).any()
torch.isinf(logits).any()

⸻

13. Summary of Part 4

* Training uses teacher forcing with shifted targets
* Loss compares predicted logits with next-token targets
* Encoder provides context once
* Decoder predicts sequence step-by-step
* Learning rate must be controlled to avoid instability

⸻

Next:

➡️ Part 5: Inference (Autoregressive Decoding) and Generation Mechanics

(including greedy decoding, loop behavior, and why full sequence is recomputed)

⸻

Continuing.

📘 Technical Report — Part 5

Inference, Autoregressive Decoding, and Generation Mechanics

⸻

1. Objective During Inference

At inference time, the model must generate output tokens without access to ground truth.

Formally:

y_t = \arg\max P(y_t \mid y_{<t}, x)

This is called autoregressive generation.

⸻

2. Initialization

generated_ids = torch.tensor([0])  # <SOS>

* <SOS> acts as the starting token
* It is the first input to the decoder

⸻

3. Encoder Execution

encoder_outputs = transformer.encoder(input_tokens)

Key property:

* Encoder runs once
* Produces fixed context:
    (seq\_len, d_{model})

⸻

4. Decoding Loop

for step in range(max_length):

Each iteration generates one new token.

⸻

5. Decoder Forward Pass

logits = transformer.decoder(generated_ids, encoder_outputs)

⸻

5.1 Input Growth

At each step:

Step	Input to Decoder
0	[<SOS>]
1	[<SOS>, w1]
2	[<SOS>, w1, w2]

⸻

5.2 Output Shape

(current_seq_len, vocab_size)

Each row = prediction for that position.

⸻

6. Selecting Next Token

next_token = argmax(logits[-1, :])

⸻

6.1 Why [-1]?

* Only the last position corresponds to the new prediction
* Earlier rows correspond to already predicted tokens

⸻

6.2 Why Argmax?

Implements greedy decoding:

y_t = \arg\max P(y_t)

Alternative strategies (not used here):

* beam search
* sampling

⸻

7. Sequence Update

generated_ids = torch.cat((generated_ids, next_token))

Sequence grows step-by-step.

⸻

8. Stop Condition

if next_token == <EOS>: break

Ensures:

* generation terminates correctly
* avoids unnecessary computation

⸻

9. Why Full Sequence is Recomputed Each Step

This is a critical design detail.

⸻

9.1 Self-Attention Dependency

Self-attention computes:

\text{Attention}(Q,K,V)

Where:

* Q depends on all tokens
* K, V depend on all tokens

Thus:

Each new token changes the attention structure for the entire sequence.

⸻

9.2 Consequence

At step t, decoder must recompute:

all previous tokens + new token

⸻

9.3 Optimization Note

Efficient implementations cache:

* K and V matrices

Your implementation recomputes everything for clarity.

⸻

10. Cross-Attention During Inference

decoder(generated_ids, encoder_outputs)

At every step:

* Query → decoder sequence
* Key, Value → encoder outputs

Thus decoder continuously:

* refines predictions
* aligns with input

⸻

11. Full Inference Flow

Input Tokens
   ↓
Encoder
   ↓
Encoder Outputs
   ↓
Initialize Decoder with <SOS>
   ↓
Loop:
   Decoder(generated_ids, encoder_outputs)
        ↓
   Select next token (argmax)
        ↓
   Append token
        ↓
   Stop if <EOS>

⸻

12. Example Walkthrough

Input:

<SOS> lets go

Expected:

<SOS> vamos <EOS>

⸻

Step-by-step

1. Start:

generated = [<SOS>]

2. Predict next:

→ vamos

3. Update:

[<SOS>, vamos]

4. Predict next:

→ <EOS>

5. Stop

⸻

13. Key Technical Observations

* Decoder uses masked self-attention → no future leakage
* Uses cross-attention → reads encoder outputs
* Uses greedy decoding → deterministic output
* Recomputes full sequence each step

⸻

14. Summary of Part 5

* Inference is autoregressive
* Decoder generates one token at a time
* Argmax selects most probable token
* Sequence grows dynamically
* Full recomputation is required due to attention

⸻

📘 Final Status

Completed

⸻

If you want next level depth, I can extend this into:

* multi-head attention derivation
* full production Transformer block
* complexity analysis (O(n²) attention)
* or optimization techniques (KV caching, FlashAttention)